# 06 — SARIMA na prática

No notebook anterior, treinamos um ARIMA simples e comparamos com uma baseline.

Agora vamos testar um modelo SARIMA.

A ideia do SARIMA é adicionar uma camada sazonal ao ARIMA, permitindo que o modelo tente capturar ciclos que se repetem ao longo do tempo.

Neste notebook, vamos:

- carregar a série diária;
- repetir a separação treino/teste;
- treinar um modelo SARIMA;
- gerar previsões;
- calcular métricas;
- comparar baseline, ARIMA e SARIMA.

### 06.01 Imports e caminhos

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_serie_diaria = raiz_projeto / "outputs" / "tabelas" / "serie_diaria_bike.csv"
caminho_metricas_arima = raiz_projeto / "outputs" / "tabelas" / "metricas_arima.csv"
caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

print("Série diária:")
print(caminho_serie_diaria)

print("\nMétricas anteriores:")
print(caminho_metricas_arima)

Série diária:
c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\serie_diaria_bike.csv

Métricas anteriores:
c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\metricas_arima.csv


### 06.02 Carregar série diária

In [2]:
serie_diaria = pd.read_csv(caminho_serie_diaria)

serie_diaria["data_hora"] = pd.to_datetime(serie_diaria["data_hora"])

serie_diaria = serie_diaria.sort_values("data_hora").reset_index(drop=True)

print(f"Linhas: {serie_diaria.shape[0]}")
print(f"Colunas: {serie_diaria.shape[1]}")

serie_diaria.head()

Linhas: 456
Colunas: 2


,data_hora,demanda
0,2011-01-01,985
1,2011-01-02,801
2,2011-01-03,1349
3,2011-01-04,1562
4,2011-01-05,1600


## Retomando a série

Vamos usar a mesma série diária trabalhada nos notebooks anteriores.

Assim, conseguimos comparar os modelos usando a mesma base e o mesmo período de teste.

### 06.03 Visualizar série

In [3]:
fig = grafico_linha_padrao(
    df=serie_diaria,
    x="data_hora",
    y="demanda",
    titulo="Demanda diária de bicicletas",
    labels={
        "data_hora": "Data",
        "demanda": "Demanda diária"
    },
    altura=500
)

fig.show()

### 06.04 Separar treino e teste

In [4]:
serie_modelagem = serie_diaria[["data_hora", "demanda"]].copy()

tamanho_teste = 60

treino = serie_modelagem.iloc[:-tamanho_teste].copy()
teste = serie_modelagem.iloc[-tamanho_teste:].copy()

print("Tamanho do treino:", treino.shape[0])
print("Tamanho do teste:", teste.shape[0])

print("\nPeríodo de treino:")
print(treino["data_hora"].min(), "até", treino["data_hora"].max())

print("\nPeríodo de teste:")
print(teste["data_hora"].min(), "até", teste["data_hora"].max())

Tamanho do treino: 396
Tamanho do teste: 60

Período de treino:
2011-01-01 00:00:00 até 2012-09-16 00:00:00

Período de teste:
2012-09-17 00:00:00 até 2012-12-19 00:00:00


## Estrutura do SARIMA

O SARIMA tem duas camadas:

- uma camada não sazonal: `(p, d, q)`;
- uma camada sazonal: `(P, D, Q, s)`.

O parâmetro `s` representa o tamanho do ciclo sazonal.

Como estamos trabalhando com uma série diária, vamos começar testando um ciclo de 7 períodos, representando um padrão semanal.

### 06.05 Definir parâmetros do SARIMA

In [5]:
order = (1, 1, 1)
seasonal_order = (1, 0, 1, 7)

print("Parâmetros não sazonais:", order)
print("Parâmetros sazonais:", seasonal_order)

Parâmetros não sazonais: (1, 1, 1)
Parâmetros sazonais: (1, 0, 1, 7)


## Treinando o modelo SARIMA

Agora vamos ajustar o modelo usando apenas o período de treino.

Depois, vamos gerar previsões para o mesmo tamanho do período de teste.

### 06.06 Treinar SARIMA

In [6]:
treino_serie = treino["demanda"].astype(float).reset_index(drop=True)

modelo_sarima = SARIMAX(
    treino_serie,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

resultado_sarima = modelo_sarima.fit(disp=False)

resultado_sarima.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                     
=========================================================================================
Dep. Variable:                           demanda   No. Observations:                  396
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 7)   Log Likelihood               -3135.194
Date:                           Fri, 26 Jun 2026   AIC                           6280.388
Time:                                   11:12:18   BIC                           6300.167
Sample:                                        0   HQIC                          6288.231
                                           - 396                                         
Covariance Type:                             opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2511      0.057      4.366      0.000       0.138       0.364
ma.L1         -0.8570      0.031    -27.950      0.000      -0.917      -0.797
ar.S.L7        0.4100      0.557      0.737      0.461      -0.681       1.501
ma.S.L7       -0.3421      0.570     -0.600      0.548      -1.459       0.775
sigma2      6.626e+05   3.58e+04     18.509      0.000    5.92e+05    7.33e+05
===================================================================================
Ljung-Box (L1) (Q):                   0.02   Jarque-Bera (JB):               109.08
Prob(Q):                              0.89   Prob(JB):                         0.00
Heteroskedasticity (H):               2.60   Skew:                            -0.72
Prob(H) (two-sided):                  0.00   Kurtosis:                         5.17
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

### 06.07 Gerar previsões

In [7]:
previsao_sarima = resultado_sarima.forecast(steps=len(teste))

teste["previsao_sarima"] = previsao_sarima.values

teste[["data_hora", "demanda", "previsao_sarima"]].head()

,data_hora,demanda,previsao_sarima
396,2012-09-17,6869,7486.740921
397,2012-09-18,4073,7571.036681
398,2012-09-19,7591,7587.662987
399,2012-10-01,6778,7572.316395
400,2012-10-02,4639,7590.249756


### 06.08 Visualizar real versus SARIMA

In [8]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=teste["data_hora"],
        y=teste["demanda"],
        mode="lines",
        name="Real",
        line=dict(color=CORES["azul_escuro"], width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=teste["data_hora"],
        y=teste["previsao_sarima"],
        mode="lines",
        name="SARIMA",
        line=dict(color=CORES["azul_principal"], width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Real versus previsto — SARIMA",
    altura=500
)

fig.show()

## Métricas de erro

Vamos usar as mesmas métricas calculadas anteriormente:

- MAE;
- RMSE;
- MAPE.

Assim conseguimos comparar o SARIMA com a baseline e com o ARIMA.

### 06.09 Função de métricas

In [9]:
def calcular_metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    y_real = np.array(y_real)
    y_pred = np.array(y_pred)

    mascara = y_real != 0
    mape = np.mean(
        np.abs((y_real[mascara] - y_pred[mascara]) / y_real[mascara])
    ) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

### 06.10 Calcular métricas do SARIMA

In [10]:
metricas_sarima = calcular_metricas(
    teste["demanda"],
    teste["previsao_sarima"]
)

df_metricas_sarima = pd.DataFrame([
    {
        "modelo": "SARIMA(1,1,1)(1,0,1,7)",
        **metricas_sarima
    }
])

df_metricas_sarima

,modelo,MAE,RMSE,MAPE
0,"SARIMA(1,1,1)(1,0,1,7)",1775.685477,2077.504788,35.530606


### 06.11 Carregar métricas anteriores

In [11]:
df_metricas_arima = pd.read_csv(caminho_metricas_arima)

df_metricas_arima

,modelo,MAE,RMSE,MAPE
0,Baseline,1585.183333,1865.908943,31.813723
1,"ARIMA(1,1,1)",1730.649703,2031.357449,34.679064


### 06.12 Comparar baseline, ARIMA e SARIMA

In [12]:
df_comparacao_modelos = pd.concat(
    [
        df_metricas_arima,
        df_metricas_sarima
    ],
    ignore_index=True
)

df_comparacao_modelos = df_comparacao_modelos.sort_values("MAPE").reset_index(drop=True)

df_comparacao_modelos

,modelo,MAE,RMSE,MAPE
0,Baseline,1585.183333,1865.908943,31.813723
1,"ARIMA(1,1,1)",1730.649703,2031.357449,34.679064
2,"SARIMA(1,1,1)(1,0,1,7)",1775.685477,2077.504788,35.530606


### 06.13 Visualizar comparação de MAPE

In [13]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=df_comparacao_modelos["modelo"],
        y=df_comparacao_modelos["MAPE"],
        marker_color=CORES["azul_principal"],
        name="MAPE"
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Comparação dos modelos pelo MAPE",
    altura=450,
    legenda_horizontal=False
)

fig.update_xaxes(title="Modelo")
fig.update_yaxes(title="MAPE (%)")

fig.show()

### 06.14 Resíduos do SARIMA

In [14]:
teste["residuo_sarima"] = teste["demanda"] - teste["previsao_sarima"]

fig = grafico_linha_padrao(
    df=teste,
    x="data_hora",
    y="residuo_sarima",
    titulo="Resíduos do modelo SARIMA no período de teste",
    labels={
        "data_hora": "Data",
        "residuo_sarima": "Erro"
    },
    altura=450
)

fig.show()

### 06.15 Resultado parcial

In [15]:
melhor_modelo = df_comparacao_modelos.iloc[0]

print("Melhor modelo nesta comparação:")
print(melhor_modelo["modelo"])

print("\nMAPE:")
print(f"{melhor_modelo['MAPE']:.2f}%")

Melhor modelo nesta comparação:
Baseline

MAPE:
31.81%


### 06.16 Salvar resultados

In [16]:
caminho_previsoes_sarima = caminho_outputs_tabelas / "previsoes_sarima.csv"
caminho_metricas_sarima = caminho_outputs_tabelas / "metricas_sarima.csv"
caminho_comparacao_modelos = caminho_outputs_tabelas / "comparacao_modelos.csv"

teste.to_csv(
    caminho_previsoes_sarima,
    index=False,
    encoding="utf-8-sig"
)

df_metricas_sarima.to_csv(
    caminho_metricas_sarima,
    index=False,
    encoding="utf-8-sig"
)

df_comparacao_modelos.to_csv(
    caminho_comparacao_modelos,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos:")
print("-", caminho_previsoes_sarima)
print("-", caminho_metricas_sarima)
print("-", caminho_comparacao_modelos)

Arquivos salvos:
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\previsoes_sarima.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\metricas_sarima.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\comparacao_modelos.csv


## Próximo passo

Treinamos o SARIMA e comparamos seu desempenho com a baseline e com o ARIMA.

No próximo notebook, vamos consolidar os resultados, escolher o modelo final e gerar uma previsão futura para comunicar o resultado do projeto.